# A: Detector (Iroda)

XLM-R base, 3 classes (unbiased / against [SIDE_A] / against [SIDE_B]) on FigNews 2024.
GPU runtime needed from A6 on (Colab: Runtime > Change runtime type > T4 GPU).
The data steps (A1-A5) run on CPU.

Code lives in `src/fignews.py`, `src/train_detector.py`, `src/metrics.py`; this notebook only calls it,
so every run uses the same split, masking, class weights and metrics.

In [2]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/debias | artifacts: /content/drive/MyDrive/debias
runtime=colab  python=3.13.16  gpu=Tesla T4  commit=0a15d96
  torch                  2.14.1+cu130
  transformers           5.23.0
  datasets               5.2.0
  accelerate             1.15.0
  bitsandbytes           0.50.2
  sentence-transformers  6.3.0
  faiss-cpu              1.16.0
  captum                 0.9.0
  scikit-learn           1.6.1
  pandas                 2.2.3
  numpy                  2.4.6


## A1-A5. Load, merge annotators, clean, split, mask

- Bias subtask only, dataset pinned to `config.FIGNEWS_REVISION`.
- A post is `(source_language, id)`; ids repeat across languages.
- Majority vote over the 7 raw labels; keep posts whose top label has >= 2 votes and a strict majority.
- Labels mapped with `config.FIGNEWS_LABEL_MAP`; "others", "both", "Unclear", "Not Applicable" dropped.
- English text = original for English posts, `english_mt` otherwise; URLs and @handles removed; duplicates dropped (original English kept).
- Test = 40% of original English posts (stratified). Val = 15% of the rest (stratified by label and language).
- Side names masked, including inside hashtags (`#IsraelHamasWar` -> `#[SIDE_A][SIDE_B]War`).

In [3]:
import json
import pandas as pd
from src import fignews, metrics, train_detector
from src.placeholders import mask_sides, find_leftovers

raw = fignews.load_raw()
posts, report = fignews.prepare(raw)
print(json.dumps({k: v for k, v in report.items() if k != "class_counts"}, indent=1, default=str))
pd.DataFrame(report["class_counts"]).T

README.md:   0%|          | 0.00/871 [00:00<?, ?B/s]

FIGNEWS-2024.tsv: reconstructing file:   0%|          |  0.00B /  196MB            

FIGNEWS-2024.tsv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

{
 "annotation_rows": 88500,
 "posts": 15000,
 "after_agreement": 9132,
 "after_label_map": 8834,
 "after_cleaning": 8827,
 "after_dedup": 8617,
 "posts_with_leftover_names": 0,
 "posts_with_russia_ukraine_names": 160,
 "posts_with_glued_names": 33,
 "glued_examples": {
  "Yisrael": 6,
  "HamasWar": 4,
  "Hamasniks": 2,
  "Fatahist,": 2,
  "Al-Habbash\u2019s": 1,
  "co.il/blogs/gazawarday108/article/sy0ukghf6": 1,
  "Gazawis": 1,
  "Hamasi": 1,
  "Hamasnik": 1,
  ".tf1info.fr/international/en-direct-guerre-[SIDE_A]-[SIDE_B]-l-onu-appelle-a-un-cease-le-feu-humanitaire-a-[SIDE_B]-une-division-israelienne-se-tire-les": 1,
  "-east/article/2024/01/23/exclusive-israels-spy-chief-proposed-[SIDE_B]-leaders-leave-[SIDE_B]-as-part-of-broader-ceasefire-talks": 1,
  "HamasAttacks": 1,
  "Hamasawi": 1,
  "Israelize": 1,
  "israelischen": 1
 },
 "test_english_only": true
}


,against_SIDE_A,against_SIDE_B,unbiased
test,60,156,450
train,587,2375,3797
val,102,419,671


In [4]:
# Leftover check: must be 0 for Israel/Palestine names.
assert report["posts_with_leftover_names"] == 0
assert report["test_english_only"]
posts.sample(8, random_state=config.SEED)[["post_id", "label", "text_masked"]]

,post_id,label,text_masked
1420,AR-5149,unbiased,🔴 [SIDE_A] helicopters drop leaflets saying th...
2283,EN-2025,unbiased,🔴 The UK has announced that it will conduct su...
6902,HE-633,against_SIDE_B,It is suspected that abductees were held in th...
5049,FR-859,against_SIDE_B,"“If we don’t stop [SIDE_B], it will arrive in ..."
4427,FR-3316,unbiased,They are finally free. After 53 days of captiv...
4988,FR-792,unbiased,[SIDE_A] Defense Minister announces 'siege' of...
6812,HE-5839,against_SIDE_B,Today (12th) there was a march in support of [...
5871,HE-1749,unbiased,"On October 7, many [SIDE_A] employees took wea..."


In [5]:
# Save the prepared posts (derived from FigNews, CC BY-SA 4.0) next to the results.
posts.to_csv(ARTIFACTS / "fignews_posts.csv", index=False)
print("class weights (train):", [round(w, 2) for w in fignews.class_weights(posts[posts.split == "train"])])

class weights (train): [0.59, 3.84, 0.95]


## A6a. Quick run on a sample (checks everything works before a long run)

In [6]:
quick = train_detector.train(posts, seed=config.SEED, sample=600, epochs=1, save_checkpoints=False, ci=False)
print(quick["thresholds"])
print(metrics.format_table({"quick (sample)": quick["test"]}))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

Map:   0%|          | 0/148 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.001495,0.236709


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'biased': 0.59, 'uncertain_low': 0.48953691124916077, 'val_macro_f1': 0.4224239117856139}
| run | n | F1 unbiased | F1 against_SIDE_A | F1 against_SIDE_B | macro-F1 |
|---|---|---|---|---|---|
| quick (sample) | 149 | 0.540 | 0.000 | 0.344 | 0.295 |


## A6b. Full runs, 3 seeds

Rough estimate 10-30 minutes per seed on a T4; write down the real time. Each run saves to
`ARTIFACTS/detector/<run>/model` with `thresholds.json`, plus `test_probs.csv` and a results JSON.
On Colab, ARTIFACTS is on Drive, so finished seeds survive a disconnect: rerun the loop and it skips them.

In [8]:
runs = []
for seed in config.SEEDS:
    done = ARTIFACTS / "results" / f"detector_xlm-roberta-base_seed{seed}.json"
    if done.exists():
        r = json.loads(done.read_text())["metrics"]
        r["model_dir"] = str(ARTIFACTS / "detector" / r["run_name"] / "model")
    else:
        r = train_detector.train(posts, seed=seed)
    runs.append(r)
    print(seed, r["thresholds"], round(r["test"]["point"]["macro_f1"], 3))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/6759 [00:00<?, ? examples/s]

Map:   0%|          | 0/1192 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Macro F1
1,0.823732,0.778092,0.680582
2,0.653429,0.726578,0.656022
3,0.513459,0.739362,0.689554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (21 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

44 {'biased': 0.58, 'uncertain_low': 0.12108474969863892, 'val_macro_f1': 0.7201166024695436} 0.631


## A7. Results: per-class F1 with 95% bootstrap CIs (test = original English posts)

Thresholds were tuned on validation only (`metrics.tune_thresholds`).

In [9]:
print(metrics.format_table({f"seed {r['seed']}": r["test"] for r in runs}))
summary = train_detector.summarize_seeds(runs)
pd.DataFrame(summary).T[["mean", "std"]].round(3)

| run | n | F1 unbiased | F1 against_SIDE_A | F1 against_SIDE_B | macro-F1 |
|---|---|---|---|---|---|
| seed 42 | 666 | 0.847 [0.821, 0.872] | 0.397 [0.296, 0.497] | 0.661 [0.592, 0.727] | 0.635 [0.588, 0.683] |
| seed 43 | 666 | 0.824 [0.795, 0.849] | 0.418 [0.333, 0.509] | 0.752 [0.696, 0.803] | 0.664 [0.626, 0.704] |
| seed 44 | 666 | 0.835 [0.808, 0.861] | 0.352 [0.259, 0.446] | 0.705 [0.640, 0.765] | 0.631 [0.586, 0.675] |


,mean,std
unbiased,0.835284,0.009637
against_SIDE_A,0.389057,0.027416
against_SIDE_B,0.706183,0.036852
macro_f1,0.643508,0.014855


In [10]:
best = max(runs, key=lambda r: r["val"]["point"]["macro_f1"])  # choose by validation, not test
print("best seed (by val):", best["seed"], best["model_dir"])
utils.save_result("detector_xlmr_summary", {"seeds": summary, "best_seed": best["seed"], "best_model_dir": best["model_dir"]})

best seed (by val): 44 /content/drive/MyDrive/debias/detector/xlm-roberta-base_seed44/model


PosixPath('/content/drive/MyDrive/debias/results/detector_xlmr_summary.json')

## A8. Captum check: top words should be framing words, not names

In [11]:
import importlib, os
from collections import Counter
os.environ["DEBIAS_DETECTOR_PATH"] = best["model_dir"]
importlib.reload(config)
from src import detector
importlib.reload(detector)

biased_test = posts[(posts.split == "test") & (posts.label != "unbiased")].sample(40, random_state=config.SEED)
words = Counter()
for t in biased_test.text_en:
    words.update(w.lower() for w in detector.explain(t))
print(words.most_common(40))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[('terrorist', 6), ('terrorists', 4), ('and', 4), ('against', 3), ('attack', 3), ('the', 3), ('by', 3), ('militants', 3), ('terror', 3), ('support', 2), ('hostages', 2), ('of', 2), ('war', 2), ('captives', 2), ('to', 2), ('a', 2), ('she', 2), ('activist', 2), ('antisemitism', 2), ('as', 2), ('released', 2), ('woman', 2), ('horrific', 1), ('majority', 1), ('getting', 1), ('that', 1), ('first', 1), ('toward', 1), ('outrage', 1), ('falling', 1), ('massive', 1), ('dead', 1), ('children', 1), ('assaults', 1), ('full', 1), ('defend', 1), ('combat', 1), ('continues', 1), ('day', 1), ('elliptic', 1)]


## Optional: upload the best model to a PRIVATE Hugging Face Hub repo
Teammates then set `DEBIAS_DETECTOR_PATH` to a local download of it (or a shared Drive folder).

In [ ]:
# from huggingface_hub import HfApi
# repo_id = "<your-hf-username>/debias-detector-xlmr"
# HfApi().create_repo(repo_id, private=True, exist_ok=True)
# HfApi().upload_folder(folder_path=best["model_dir"], repo_id=repo_id)



```
# This is formatted as code
```

## Bias checkpoint: score Madina's baseline answers
Runs when `data/base_answers.csv` exists (column with the answer text).

In [12]:
base_path = config.DATA_DIR / "base_answers.csv"
if base_path.exists():
    base = pd.read_csv(base_path)
    text_col = next(c for c in ("answer", "base_answer", "text", "response") if c in base.columns)
    conflict_col = base["conflict"] if "conflict" in base.columns else config.DEFAULT_CONFLICT
    scored = pd.DataFrame([detector.detect(t, c) for t, c in zip(base[text_col], pd.Series(conflict_col, index=base.index))])
    base = pd.concat([base, scored], axis=1)
    print(base["status"].value_counts(normalize=True).round(3))
    base.to_csv(ARTIFACTS / "base_answers_scored.csv", index=False)
else:
    print("data/base_answers.csv not there yet")

data/base_answers.csv not there yet


## A9. Second detector (mDeBERTa-v3), different split, evaluation only
Never used inside the pipeline or for training anything else.

In [14]:
posts2, report2 = fignews.prepare(raw, seed=config.SEED + 1000)  # different split
second = train_detector.train(posts2, seed=config.SEED, model_name=config.SECOND_DETECTOR_BASE_MODEL)
print(metrics.format_table({"mdeberta (split 2)": second["test"]}))

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
classifier.bias     

Map:   0%|          | 0/6759 [00:00<?, ? examples/s]

Map:   0%|          | 0/1192 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


ValueError: Attempting to unscale FP16 gradients.